# 65 - Signal Sparsity Analysis: Is the Signal Always Firing?

**Problem:** Continuous signals that are always "on" are hard to extract alpha from because:
1. No clear entry/exit points
2. Constant position adjustments = overtrading
3. Signal noise drowns out the real information

**Solution:** Make the signal SPARSE - only fire on high-conviction moments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

data = {m: load_metric(m) for m in ['price', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'sopr', 'aviv']}
print("Data loaded")

In [ ]:
# Build composite signal
CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

for metric in CONFIG.keys():
    if metric in data and not data[metric].empty:
        df = df.join(data[metric][['value']].rename(columns={'value': metric}), how='left')
        df[metric] = df[metric].ffill()

def calc_composite(row):
    total_score, total_weight = 0, 0
    for metric, cfg in CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['signal'] = df.apply(calc_composite, axis=1)
print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")

## Part 1: How Often is the Signal "Firing"?

In [ ]:
print("SIGNAL DISTRIBUTION ANALYSIS")
print("="*60)

print(f"\nSignal Statistics:")
print(f"  Min:    {df['signal'].min():.2f}")
print(f"  Max:    {df['signal'].max():.2f}")
print(f"  Mean:   {df['signal'].mean():.2f}")
print(f"  Median: {df['signal'].median():.2f}")
print(f"  Std:    {df['signal'].std():.2f}")

zones = [
    ('Extreme Bullish', df['signal'] < -1.5),
    ('Bullish', (df['signal'] >= -1.5) & (df['signal'] < -0.5)),
    ('Neutral', (df['signal'] >= -0.5) & (df['signal'] <= 0.5)),
    ('Bearish', (df['signal'] > 0.5) & (df['signal'] <= 1.5)),
    ('Extreme Bearish', df['signal'] > 1.5),
]

print(f"\nTime Spent in Each Zone:")
print("-"*50)
for name, mask in zones:
    pct = mask.sum() / len(df) * 100
    days = mask.sum()
    print(f"  {name:<20}: {pct:>5.1f}% ({days:>4} days)")

neutral_pct = ((df['signal'] >= -0.5) & (df['signal'] <= 0.5)).sum() / len(df) * 100
extreme_pct = ((df['signal'] < -1.0) | (df['signal'] > 1.0)).sum() / len(df) * 100

print(f"\n" + "="*60)
print(f"KEY INSIGHT:")
print(f"  Signal in 'neutral' zone:  {neutral_pct:.1f}% of the time")
print(f"  Signal at extremes:        {extreme_pct:.1f}% of the time")
print(f"\n  → The signal is mostly in the MIDDLE where it's not useful!")
print("="*60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['signal'], bins=50, color='#3b82f6', edgecolor='white', alpha=0.7)
axes[0].axvline(x=-1.0, color='#22c55e', linestyle='--', linewidth=2, label='Buy zone (<-1)')
axes[0].axvline(x=1.0, color='#ef4444', linestyle='--', linewidth=2, label='Sell zone (>1)')
axes[0].axvline(x=0, color='white', linestyle='-', alpha=0.3)
axes[0].set_xlabel('Signal Value')
axes[0].set_ylabel('Frequency (days)')
axes[0].set_title('Signal Distribution: Most Time in the Middle!')
axes[0].legend()

axes[1].plot(df.index, df['signal'], color='#3b82f6', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=-1.0, color='#22c55e', linestyle='--', linewidth=2)
axes[1].axhline(y=1.0, color='#ef4444', linestyle='--', linewidth=2)
axes[1].fill_between(df.index, -1.0, df['signal'].min(), alpha=0.2, color='#22c55e', label='Buy zone')
axes[1].fill_between(df.index, 1.0, df['signal'].max(), alpha=0.2, color='#ef4444', label='Sell zone')
axes[1].set_ylabel('Signal')
axes[1].set_title('Signal Over Time: Only Extremes Matter')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## Part 2: Forward Returns by Signal Level

In [ ]:
for days in [7, 30, 60, 90, 180]:
    df[f'fwd_{days}d'] = df['price'].shift(-days) / df['price'] - 1

df['signal_decile'] = pd.qcut(df['signal'], 10, labels=False, duplicates='drop')

print("\nFORWARD RETURNS BY SIGNAL DECILE")
print("="*70)
print(f"{'Decile':<10} {'Signal Range':<20} {'30d Fwd':>12} {'90d Fwd':>12}")
print("-"*70)

for decile in range(10):
    mask = df['signal_decile'] == decile
    sig_min = df.loc[mask, 'signal'].min()
    sig_max = df.loc[mask, 'signal'].max()
    ret_30 = df.loc[mask, 'fwd_30d'].mean() * 100
    ret_90 = df.loc[mask, 'fwd_90d'].mean() * 100
    print(f"  {decile:<8} [{sig_min:>5.2f}, {sig_max:>5.2f}]      {ret_30:>+10.1f}% {ret_90:>+10.1f}%")

print(f"\n→ Only the EXTREMES have predictive power!")

## Part 3: Creating a SPARSE Signal

In [ ]:
def create_sparse_signal(signal, buy_threshold=-1.0, sell_threshold=1.0):
    """Convert continuous signal to sparse: 1 (BUY), -1 (SELL), or 0 (NO SIGNAL)"""
    if signal <= buy_threshold: return 1
    elif signal >= sell_threshold: return -1
    else: return 0

thresholds = [(-0.5, 0.5), (-0.75, 0.75), (-1.0, 1.0), (-1.25, 1.25), (-1.5, 1.5)]

print("\nSPARSE SIGNAL ANALYSIS")
print("="*70)
print(f"{'Thresholds':<15} {'BUY %':>10} {'SELL %':>10} {'NO SIG %':>10} {'Signal Days':>12}")
print("-"*70)

for buy_t, sell_t in thresholds:
    sparse = df['signal'].apply(lambda x: create_sparse_signal(x, buy_t, sell_t))
    buy_pct = (sparse == 1).sum() / len(sparse) * 100
    sell_pct = (sparse == -1).sum() / len(sparse) * 100
    no_sig_pct = (sparse == 0).sum() / len(sparse) * 100
    signal_days = (sparse != 0).sum()
    print(f"  [{buy_t:>5.2f}, {sell_t:>5.2f}]   {buy_pct:>8.1f}%  {sell_pct:>8.1f}%  {no_sig_pct:>8.1f}%  {signal_days:>10}")

In [ ]:
BUY_THRESHOLD = -1.0
SELL_THRESHOLD = 1.0

df['sparse_signal'] = df['signal'].apply(lambda x: create_sparse_signal(x, BUY_THRESHOLD, SELL_THRESHOLD))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].semilogy(df.index, df['price'], color='white', linewidth=1, alpha=0.7)
buy_mask = df['sparse_signal'] == 1
for i in range(len(df)-1):
    if buy_mask.iloc[i]:
        axes[0].axvspan(df.index[i], df.index[i+1], alpha=0.3, color='#22c55e')
sell_mask = df['sparse_signal'] == -1
for i in range(len(df)-1):
    if sell_mask.iloc[i]:
        axes[0].axvspan(df.index[i], df.index[i+1], alpha=0.3, color='#ef4444')
axes[0].set_ylabel('Price (USD)')
axes[0].set_title(f'Sparse Signal: Only Fire at Extremes', fontsize=14, fontweight='bold')

axes[1].fill_between(df.index, 0, df['sparse_signal'], 
                      where=df['sparse_signal'] > 0, color='#ef4444', alpha=0.7, label='SELL')
axes[1].fill_between(df.index, 0, df['sparse_signal'], 
                      where=df['sparse_signal'] < 0, color='#22c55e', alpha=0.7, label='BUY')
axes[1].set_ylabel('Sparse Signal')
axes[1].legend()

plt.tight_layout()
plt.show()

signal_pct = (df['sparse_signal'] != 0).sum() / len(df) * 100
print(f"\nSignal fires {signal_pct:.1f}% of the time")

## Summary

**PROBLEM:** Continuous signals are always "on" - mostly in neutral zone with no predictive power.

**SOLUTION:** Make signal SPARSE - only fire at extremes.

```python
def sparse_signal(score):
    if score < -1.0:   return "BUY"
    elif score > 1.0:  return "SELL"  
    else:              return "NO SIGNAL"  # Do nothing!
```

**KEY INSIGHT:** Good trading signals are SPARSE. They fire rarely but with high conviction.